# Zero-shot baseline audit, Gemma 4 E4B on civic data visualizations

Pre-fine-tune audit of the base model on **27 civic data visualizations**: 9 real-world dashboards and 18 synthetic charts (6 stacked bars, 6 box plots, 6 choropleths). The original audit covered 28 images; one source file (`choropleth-paris-detailed.png`) was lost from the repo and is no longer reproducible.

Pattern: load the published fine-tuned adapter (`shahfazal/civicinsight-gemma4-e4b-it@v1.0`) and run inference inside `with model.disable_adapter()` so the same loaded model produces base-Gemma-4 zero-shot output. Identical to Demo 0 in the canonical Kaggle submission notebook.

**How to run.** Open this notebook in [Modal Notebooks](https://modal.com), attach an A100-40GB GPU and the `civicinsight-data` volume at `/mnt/civicinsight`, then Run All. Each phase cell is independently re-runnable. Total runtime ~25 min at `max_new_tokens=1024`.

**Image data.** The cells expect the audit images at `/mnt/civicinsight/audit/` on the `civicinsight-data` volume. Upload from local repo once via:

```
modal volume put civicinsight-data examples/standardized /audit/examples/standardized
modal volume put civicinsight-data test_charts /audit/test_charts
```

Findings distilled from these outputs are summarised in [`docs/zero-shot-evaluation-report.md`](https://github.com/shahfazal/civicinsight/blob/main/docs/zero-shot-evaluation-report.md).


## 1. Install pinned stack

In [1]:
%%capture
# Same pin set as the canonical submission notebook (matches the v1 adapter).
!pip install -q --upgrade \
    unsloth==2026.4.8 \
    unsloth-zoo==2026.4.9 \
    transformers==5.5.0 \
    peft==0.19.1 \
    bitsandbytes==0.49.2 \
    pillow==11.3.0


## 2. Authenticate with HuggingFace

The v1 adapter is private until the May 13 public flip. Pass an HF token with read access to `shahfazal/civicinsight-gemma4-e4b-it`. Set `HF_TOKEN` as a Modal Secret in your Notebook settings, or paste interactively.


In [2]:
import os
from huggingface_hub import login

if os.environ.get("HF_TOKEN"):
    login(token=os.environ["HF_TOKEN"])
    print("Logged in via HF_TOKEN env var.")
else:
    login()  # interactive prompt


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Logged in via HF_TOKEN env var.


## 3. Load the fine-tuned adapter

Loads the LoRA-on-Gemma-4-E4B 4-bit checkpoint from HuggingFace Hub. Subsequent zero-shot inference disables the adapter so the base model answers; the adapter stays loaded so you can A/B compare in the same session if you want.


In [3]:
from unsloth import FastVisionModel
import torch

model, tokenizer = FastVisionModel.from_pretrained(
    "shahfazal/civicinsight-gemma4-e4b-it",
    revision="v1.0",
    load_in_4bit=True,
)
FastVisionModel.for_inference(model)
print("FT adapter loaded. Zero-shot runs via `with model.disable_adapter()`.")
print(f"GPU 0 free: {torch.cuda.mem_get_info(0)[0]/1e9:.1f}GB")


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.4.8: Fast Gemma4 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 1. Max memory: 79.251 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/2130 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/203 [00:00<?, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/32.2M [00:00<?, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/158M [00:00<?, ?B/s]

/usr/local/lib/python3.12/site-packages/peft/peft_model.py:622: UserWarning: Found missing adapter keys while loading the checkpoint: ['base_model.model.model.language_model.layers.24.self_attn.k_proj.lora_A.default.weight', 'base_model.model.model.language_model.layers.24.self_attn.k_proj.lora_B.default.weight', 'base_model.model.model.language_model.layers.24.self_attn.v_proj.lora_A.default.weight', 'base_model.model.model.language_model.layers.24.self_attn.v_proj.lora_B.default.weight', 'base_model.model.model.language_model.layers.25.self_attn.k_proj.lora_A.default.weight', 'base_model.model.model.language_model.layers.25.self_attn.k_proj.lora_B.default.weight', 'base_model.model.model.language_model.layers.25.self_attn.v_proj.lora_A.default.weight', 'base_model.model.model.language_model.layers.25.self_attn.v_proj.lora_B.default.weight', 'base_model.model.model.language_model.layers.26.self_attn.k_proj.lora_A.default.weight', 'base_model.model.model.language_model.layers.26.self_a

FT adapter loaded. Zero-shot runs via `with model.disable_adapter()`.
GPU 0 free: 73.4GB


## 4. Inference helper

`max_new_tokens` defaults to 1024 for ~50s per image. Bump to 2048 if you want fully verbose outputs on the most complex dashboards (tourisme-powerbi, dense choropleths) at the cost of ~2 min/image.


In [4]:
import time
from pathlib import Path
from PIL import Image

# Audit images live on the civicinsight-data volume under /audit/.
# See header for the upload commands.
DATA_DIR = Path("/mnt/civicinsight/audit")
MAX_NEW_TOKENS = 1024

PROMPT = "Describe this data visualization.\nExtract all visible numbers, percentages, and values.\nUse this structure:\n\n## Chart Type\n[what type of chart is this]\n\n## Key Values\n[list every number, percentage, or data value you can see]\n\n## Trends\n[any patterns or insights visible]"


def test_dashboard(rel_path, label):
    image = Image.open(DATA_DIR / rel_path)
    messages = [{
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": PROMPT},
        ],
    }]
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    ).to(model.device)

    start = time.time()
    with model.disable_adapter():
        outputs = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS)
    elapsed = time.time() - start

    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
    print(f"\n{'='*60}")
    print(f"IMAGE: {label} | Time: {elapsed:.1f}s")
    print(f"{'='*60}")
    print(decoded)


## 5. Phase 1, real-world dashboards (9 images)

_The original audit covered 10 real-world images. `choropleth-paris-detailed.png` was lost from the repo and is no longer reproducible; its qualitative findings remain documented in `docs/zero-shot-evaluation-report.md`._


In [5]:
real_world = {
    "tourisme_powerbi":           "examples/standardized/tourisme-powerbi.png",
    "scatter_all_parties":        "examples/standardized/scatter-all-parties-clean.png",
    "scatter_gauche_filtered":    "examples/standardized/scatter-gauche-filtered.png",
    "scatter_gauche_droite":      "examples/standardized/scatter-gauche-droite-filtered.png",
    "scatter_point_selected":     "examples/standardized/scatter-point-selected.png",
    "quintile_distrib":           "examples/standardized/quintile-distrib.png",
    "boxplot_abstention":         "examples/standardized/boxplot-abstention.png",
    "choropleth_paris":           "examples/standardized/choropleth-paris.png",
    "choropleth_marseille":       "examples/standardized/choropleth-marseille.png",
}
for label, path in real_world.items():
    test_dashboard(path, label)



IMAGE: tourisme_powerbi | Time: 192.4s
user
Describe this data visualization.
Extract all visible numbers, percentages, and values.
Use this structure:

## Chart Type
[what type of chart is this]

## Key Values
[list every number, percentage, or data value you can see]

## Trends
[any patterns or insights visible]
model
## Chart Type
This image contains several data visualizations, including:
1. A table showing tourism data by month and year.
2. A bar chart showing tourism expenditure and its relation to GDP by region.
3. A map visualization showing tourist itineraries.
4. A gauge chart showing a "Score touristique".

## Key Values
**Tourism Data Table (Seasonality):**
* **Region:** Auvergne-Rhône-Alpes
* **Total Arrivals (2024 vs 2023):** 14.6M (2.3% increase)
* **Total Arrivals (2024):** 2024 vs 2023 (Implied value)
* **Total Arrivals (2023):** 2023 vs 2022 (Implied value)
* **Monthly Data (Arrivals - implied units):**
    * Janvier: 1,647,000 (2022), 2,122,000 (2023), 2,011,000 (20

## 6. Phase 2a, synthetic stacked bars (6 images)

In [6]:
stacked_bars = {
    "stacked_bar_bars_3_001":         "test_charts/stacked_bar_bars_3_001.png",
    "stacked_bar_bars_8_003":         "test_charts/stacked_bar_bars_8_003.png",
    "stacked_bar_dist_even_007":      "test_charts/stacked_bar_dist_even_007.png",
    "stacked_bar_dist_dominant_008":  "test_charts/stacked_bar_dist_dominant_008.png",
    "stacked_bar_dist_tiny_010":      "test_charts/stacked_bar_dist_tiny_010.png",
    "stacked_bar_random_011":         "test_charts/stacked_bar_random_011.png",
}
for label, path in stacked_bars.items():
    test_dashboard(path, label)



IMAGE: stacked_bar_bars_3_001 | Time: 102.6s
user
Describe this data visualization.
Extract all visible numbers, percentages, and values.
Use this structure:

## Chart Type
[what type of chart is this]

## Key Values
[list every number, percentage, or data value you can see]

## Trends
[any patterns or insights visible]
model
## Chart Type
Stacked Bar Chart

## Key Values
**Y-axis Labels (Categories/Groups):**
* Prix 1 864,0-2 064,0 €/m²
* Prix 2 906,0-3 106,0 €/m²
* Prix 2 343,0-2 543,0 €/m²

**X-axis Values (Percentages):**
* 0, 20, 40, 60, 80, 100

**Legend/Data Series (Colors):**
* Extrême gauche (Red/Pink)
* Gauche (Pink/Light Red)
* Centre (Yellow)
* Divers (Grey)
* Droite (Blue)
* Extrême droite (Black)

**Approximate Data Distribution (Reading from the bars):**

**Prix 1 864,0-2 064,0 €/m²:**
* Extrême gauche: $\approx 40\%$
* Gauche: $\approx 30\%$
* Centre: $\approx 15\%$
* Divers: $\approx 10\%$
* Droite: $\approx 15\%$
* Extrême droite: $\approx 10\%$

**Prix 2 906,0-3 106

## 7. Phase 2b, synthetic box plots (6 images)

In [7]:
box_plots = {
    "boxplot_01_baseline":         "test_charts/boxplots/boxplot_01_baseline.png",
    "boxplot_02_outliers":         "test_charts/boxplots/boxplot_02_outliers.png",
    "boxplot_03_tooltips_center":  "test_charts/boxplots/boxplot_03_tooltips_center.png",
    "boxplot_04_proximity":        "test_charts/boxplots/boxplot_04_proximity.png",
    "boxplot_05_size_variation":   "test_charts/boxplots/boxplot_05_size_variation.png",
    "boxplot_06_full_complexity":  "test_charts/boxplots/boxplot_06_full_complexity.png",
}
for label, path in box_plots.items():
    test_dashboard(path, label)



IMAGE: boxplot_01_baseline | Time: 91.4s
user
Describe this data visualization.
Extract all visible numbers, percentages, and values.
Use this structure:

## Chart Type
[what type of chart is this]

## Key Values
[list every number, percentage, or data value you can see]

## Trends
[any patterns or insights visible]
model
## Chart Type
Box Plot

## Key Values
**Y-axis (Note sur 20):**
* 0
* 5
* 10
* 15
* 20

**Data Points (Approximate Median/Quartiles/Whiskers):**

**Maths:**
* Lower Whisker: $\approx 7.5$
* Q1: $\approx 11.5$
* Median: $\approx 13$
* Q3: $\approx 14.5$
* Upper Whisker: $\approx 17.5$

**Français:**
* Lower Whisker: $\approx 6.5$
* Q1: $\approx 9.5$
* Median: $\approx 11$
* Q3: $\approx 12.5$
* Upper Whisker: $\approx 15$

**Histoire:**
* Lower Whisker: $\approx 7$
* Q1: $\approx 13$
* Median: $\approx 15$
* Q3: $\approx 16$
* Upper Whisker: $\approx 19.5$

**Sciences:**
* Lower Whisker: $\approx 7.5$
* Q1: $\approx 9$
* Median: $\approx 10.5$
* Q3: $\approx 15$
* Upp

## 8. Phase 2c, synthetic choropleths (6 images)

In [8]:
choropleths = {
    "choropleth_01_political":          "test_charts/choropleths/choropleth_01_political.png",
    "choropleth_02_transport":          "test_charts/choropleths/choropleth_02_transport.png",
    "choropleth_03_circles":            "test_charts/choropleths/choropleth_03_circles.png",
    "choropleth_04_political_circles":  "test_charts/choropleths/choropleth_04_political_circles.png",
    "choropleth_05_gradient":           "test_charts/choropleths/choropleth_05_gradient.png",
    "choropleth_06_gradient_labels":    "test_charts/choropleths/choropleth_06_gradient_labels.png",
}
for label, path in choropleths.items():
    test_dashboard(path, label)



IMAGE: choropleth_01_political | Time: 41.3s
user
Describe this data visualization.
Extract all visible numbers, percentages, and values.
Use this structure:

## Chart Type
[what type of chart is this]

## Key Values
[list every number, percentage, or data value you can see]

## Trends
[any patterns or insights visible]
model
## Chart Type
This is a conceptual diagram or infographic, not a standard data chart (like a bar chart or pie chart). It uses colored, hexagonal shapes to represent different "Zones" (A through L), which are categorized by their winning political bloc.

## Key Values
There are no visible numbers, percentages, or explicit data values within the visualization itself. The values are represented by the labels of the zones (Zone A, Zone B, etc.) and the associated color coding for the political blocs.

## Trends
The visualization shows a distribution of political blocs across 12 distinct zones. The color coding indicates which bloc won in each zone:

*   **Red/Pink Hu

## 9. Cleanup

In [9]:
import gc
import torch

del model
gc.collect()
torch.cuda.empty_cache()
print("Memory cleared.")
print(f"GPU 0 free: {torch.cuda.mem_get_info(0)[0]/1e9:.1f}GB")


Memory cleared.
GPU 0 free: 84.5GB
